# ALV Master Replication Notebook
**Study:** Artificial Linguistic Veneer (ALV), AI Agency (AIA), and Oral Defense Performance (ODP)

This notebook is a complete, structured analysis shell for the study package (templates, not empirical results).
**Pipeline:** Setup → Load data → Score scales → Descriptives → Reliability → Correlations → Assumptions → Regression (moderation) → Simple slopes → Export tables & figures.
All templates live next to this notebook in `/mnt/data/`.

## 1. Setup
Imports, global settings, and paths. No results are computed here.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Optional packages (guarded so the notebook still runs without them)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    HAVE_SM = True
except ImportError:
    HAVE_SM = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "figure.autolayout": True})

DATA = "/mnt/data"
PATHS = {
    "raw":          os.path.join(DATA, "raw_data_template.csv"),
    "codebook":     os.path.join(DATA, "codebook.csv"),
    "desc":         os.path.join(DATA, "descriptive_statistics_template.csv"),
    "corr":         os.path.join(DATA, "correlation_matrix_template.csv"),
    "rel":          os.path.join(DATA, "reliability_template.csv"),
    "reg":          os.path.join(DATA, "regression_results_template.csv"),
    "slopes":       os.path.join(DATA, "simple_slopes_template.csv"),
    "assumptions":  os.path.join(DATA, "assumptions_checklist.csv"),
    "figures":      os.path.join(DATA, "figures_inventory.csv"),
    "workbook":     os.path.join(DATA, "research_workbook_template.xlsx"),
}
print("statsmodels available:", HAVE_SM)
print("Files present:", {k: os.path.exists(p) for k, p in PATHS.items()})

## 2. Load data & codebook
Replace `raw_data_template.csv` with the real data file when available; the rest of the pipeline is unchanged.

In [ ]:
codebook = pd.read_csv(PATHS["codebook"])
raw = pd.read_csv(PATHS["raw"])

ITEMS = {
    "alv": [f"alv{i}" for i in range(1, 6)],
    "aia": [f"aia{i}" for i in range(1, 6)],
    "odp": [f"odp{i}" for i in range(1, 5)],
}
META = ["id", "gender", "age", "major", "english_level"]

print(f"raw shape: {raw.shape}")
raw.head()

## 3. Data preparation
- Type coercion for numeric/categorical columns
- Range checks (Likert 1–5, rubric 1–? adjust as needed)
- Composite scores (mean of items; change `MEAN_OF_ITEMS` to use sums)

In [ ]:
LIKERT_RANGE = (1, 5)
MEAN_OF_ITEMS = True

for col in ["age"] + sum(ITEMS.values(), []):
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

for col in ["gender", "major", "english_level"]:
    raw[col] = raw[col].astype("category")

def range_report(df, items, lo, hi):
    vals = df[items].stack()
    return {"n_valid": int(vals.notna().sum()),
            "out_of_range": int(((vals < lo) | (vals > hi)).sum())}

print("ALV:", range_report(raw, ITEMS["alv"], *LIKERT_RANGE))
print("AIA:", range_report(raw, ITEMS["aia"], *LIKERT_RANGE))

def composite(df, items):
    s = df[items].mean(axis=1) if MEAN_OF_ITEMS else df[items].sum(axis=1)
    return s.where(df[items].notna().sum(axis=1) >= 0.8 * len(items))

for scale in ITEMS:
    name = f"{scale}_total"
    raw[name] = composite(raw, ITEMS[scale])
    if name not in raw.columns and False: pass
raw["odp_total"] = composite(raw, ITEMS["odp"])

raw[["alv_total", "aia_total", "odp_total"]].describe()

## 4. Descriptive statistics
M, SD, min, max, skewness, kurtosis, n for all key variables. Mirrors `descriptive_statistics_template.csv`.

In [ ]:
def describe_vars(df, vars_):
    rows = []
    for v in vars_:
        s = df[v].dropna().astype(float)
        rows.append({
            "Variable": v, "N": len(s), "Mean": s.mean(), "SD": s.std(ddof=1),
            "Min": s.min(), "Max": s.max(),
            "Skewness": stats.skew(s, bias=False) if len(s) > 2 else np.nan,
            "Kurtosis": stats.kurtosis(s, bias=False) if len(s) > 3 else np.nan,
        })
    return pd.DataFrame(rows).round(3)

key_vars = ["age", "alv_total", "aia_total", "odp_total"]
desc = describe_vars(raw, key_vars)
print("NOTE: empty templates -> values are NaN until real data are entered.")
desc

## 5. Reliability (Cronbach's alpha)
Internal consistency for ALV, AIA, and ODP scales, matching `reliability_template.csv`.

In [ ]:
def cronbach_alpha(df, items):
    d = df[items].dropna()
    k = len(items)
    if len(d) < 2 or k < 2 or d.var(ddof=1).sum() == 0:
        return np.nan
    return k / (k - 1) * (1 - d.var(ddof=1).sum() / d.sum(axis=1).var(ddof=1))

rel = pd.DataFrame([
    {"Scale": s.upper(), "Items": k, "n_complete": int(raw[ITEMS[s]].dropna().shape[0]),
     "Cronbach_alpha": round(cronbach_alpha(raw, ITEMS[s]), 3) if raw[ITEMS[s]].notna().any().any() else np.nan}
    for s, k in [("alv", 5), ("aia", 5), ("odp", 4)]
])
rel

## 6. Correlation matrix
Pearson r (Spearman alternative noted) for the key continuous variables; mirrors `correlation_matrix_template.csv`.

In [ ]:
# Use method="spearman" if normality is violated (see Section 7).
corr = raw[key_vars].corr(method="pearson").round(3)
print(corr)

# Significance p-values (pairwise, listwise-deleted per pair)
def corr_pvalues(df, vars_):
    n = len(df)
    p = pd.DataFrame(np.ones((len(vars_), len(vars_))), index=vars_, columns=vars_)
    for i, a in enumerate(vars_):
        for b in vars_[i+1:]:
            sub = df[[a, b]].dropna()
            if len(sub) > 2:
                r, pv = stats.pearsonr(sub[a], sub[b])
                p.loc[a, b] = p.loc[b, a] = pv
    return p
print("\np-values:")
print(corr_pvalues(raw, key_vars).round(4))

## 7. Assumptions checklist
Normality, linearity, homoscedasticity, multicollinearity — the structure of `assumptions_checklist.csv`.
Runs only when real data exist; prints status otherwise.

In [ ]:
def check_assumptions(df, y="odp_total", xs=("alv_total", "aia_total")):
    d = df[[y, *xs]].dropna()
    if len(d) < 10:
        print("Insufficient data — fill raw_data_template.csv first.")
        return None
    checks = {}
    for v in [y, *xs]:
        w = stats.shapiro(d[v]) if len(d) <= 5000 else stats.normaltest(d[v])
        checks[f"Normality {v}"] = f"W={w[0]:.3f}, p={w[1]:.3f}"
    X = sm.add_constant(d[list(xs)]) if HAVE_SM else None
    if HAVE_SM:
        model = sm.OLS(d[y], X).fit()
        from statsmodels.stats.diagnostic import het_breuschpagan
        from statsmodels.stats.outliers_influence import variance_inflation_factor
        bp = het_breuschpagan(model.resid, X)
        checks["Homoscedasticity (Breusch-Pagan)"] = f"LM={bp[0]:.3f}, p={bp[1]:.3f}"
        for i, v in enumerate(list(xs), start=1):
            checks[f"VIF {v}"] = f"{variance_inflation_factor(X, i):.2f}"
    return checks

check_assumptions(raw)

## 8. Moderation regression
Model: `ODP ~ ALV + AIA + ALV×AIA` (mean-centered predictors).
Populates the structure of `regression_results_template.csv`: B, SE, Beta, t, p, 95% CI.

In [ ]:
if not HAVE_SM:
    print("statsmodels not installed — install it to run this section.")
else:
    d = raw[["odp_total", "alv_total", "aia_total"]].dropna().copy()
    for v in ["alv_total", "aia_total"]:
        d[f"{v}_c"] = d[v] - d[v].mean()
    d["interaction"] = d["alv_total_c"] * d["aia_total_c"]

    if len(d) >= 10:
        model = smf.ols("odp_total ~ alv_total_c + aia_total_c + interaction", data=d).fit()
        reg_table = pd.DataFrame({
            "Predictor": ["Intercept", "ALV total", "AI Agency total", "ALV x AI Agency"],
            "B": model.params.values, "SE": model.bse.values,
            "t": model.tvalues.values, "p": model.pvalues.values,
            "CI_low": model.conf_int()[0].values, "CI_high": model.conf_int()[1].values,
        })
        reg_table["Beta"] = [np.nan] + list(
            (pd.DataFrame({"x": d[["alv_total_c", "aia_total_c", "interaction"]],
                           "y": d["odp_total"}).corr().loc[["alv_total_c", "aia_total_c", "interaction"], "odp_total"]
             ).values)
        print(model.summary())
        print("\nR2 = {:.3f}, adj. R2 = {:.3f}, F = {:.2f}, p = {:.4g}".format(
            model.rsquared, model.rsquared_adj, model.fvalue, model.f_pvalue))
    else:
        print("Insufficient rows (need >= 10 complete cases); template stays empty.")
        reg_table = pd.read_csv(PATHS["reg"])

## 9. Simple slopes (probe the interaction)
Slopes of ALV on ODP at AIA = M − 1SD, M, M + 1SD. Mirrors `simple_slopes_template.csv`.

In [ ]:
if HAVE_SM and "model" in dir():
    slopes = []
    for label, z in [("Low (-1SD)", -1), ("Mean", 0), ("High (+1SD)", 1)]:
        d2 = d.copy()
        d2["aia_mod"] = d2["aia_total_c"] - z * d2["aia_total_c"].std(ddof=1)
        d2["alv_x_aia"] = d2["alv_total_c"] * d2["aia_mod"]
        m = smf.ols("odp_total ~ alv_total_c + aia_mod + alv_x_aia", data=d2).fit()
        slopes.append({
            "Level of AIA": label,
            "Slope of ALV (B)": m.params["alv_total_c"],
            "SE": m.bse["alv_total_c"], "t": m.tvalues["alv_total_c"], "p": m.pvalues["alv_total_c"],
        })
    slopes_df = pd.DataFrame(slopes).round(4)
else:
    print("Run Section 8 with real data first; see simple_slopes_template.csv.")
    slopes_df = pd.read_csv(PATHS["slopes"])
slopes_df

## 10. Figures
Scatterplot with fitted line and interaction plot. Save under `/mnt/data/figures/` and log to `figures_inventory.csv`.

In [ ]:
import os as _os
FIG_DIR = os.path.join(DATA, "figures"); _os.makedirs(FIG_DIR, exist_ok=True)
inventory = pd.read_csv(PATHS["figures"])
made = []

d_plot = raw[["alv_total", "odp_total"]].dropna()
if len(d_plot) > 1:
    plt.figure(figsize=(6, 4))
    plt.scatter(d_plot["alv_total"], d_plot["odp_total"], alpha=0.7)
    b1, b0 = np.polyfit(d_plot["alv_total"], d_plot["odp_total"], 1)
    xs_ = np.linspace(d_plot["alv_total"].min(), d_plot["alv_total"].max(), 50)
    plt.plot(xs_, b0 + b1 * xs_, "r--", label=f"b = {b1:.2f}")
    plt.xlabel("ALV total"); plt.ylabel("ODP total"); plt.title("ALV vs ODP"); plt.legend()
    f1 = os.path.join(FIG_DIR, "fig1_alv_odp_scatter.png")
    plt.savefig(f1); plt.show(); made.append(("Figure 1", f1))
else:
    print("No data yet — Figure 1 skipped.")

if HAVE_SM and "slopes_df" in dir() and len(d) >= 10:
    lo, hi = d["aia_total_c"].mean() - d["aia_total_c"].std(), d["aia_total_c"].mean() + d["aia_total_c"].std()
    plt.figure(figsize=(6, 4))
    for aia_val, lab in [(lo, "Low AIA (-1SD)"), (d["aia_total_c"].mean(), "Mean AIA"), (hi, "High AIA (+1SD)")]:
        sub = d.copy()
        sub["aia_fix"] = aia_val
        sub["alv_x_aia"] = sub["alv_total_c"] * (sub["aia_fix"] - sub["aia_total_c"])
        pred = model.params["Intercept"] + model.params["alv_total_c"] * sub["alv_total_c"]              + model.params["aia_total_c"] * (aia_val - d["aia_total_c"].mean())              + model.params["interaction"] * sub["alv_x_aia"] / (aia_val - d["aia_total_c"].mean() if aia_val != d["aia_total_c"].mean() else 1)
        plt.plot(sub["alv_total_c"], pred, label=lab)
    plt.xlabel("ALV (centered)"); plt.ylabel("Predicted ODP"); plt.title("Interaction: ALV x AIA"); plt.legend()
    f2 = os.path.join(FIG_DIR, "fig2_interaction.png")
    plt.savefig(f2); plt.show(); made.append(("Figure 2", f2))

if made:
    new_rows = pd.DataFrame([{"Figure": n, "Path": p, "Status": "generated"} for n, p in made])
    inventory = pd.concat([inventory, new_rows], ignore_index=True) if len(inventory) else new_rows
    inventory.to_csv(os.path.join(DATA, "figures_inventory_updated.csv"), index=False)
    print("Inventory updated:", [p for _, p in made])

## 11. Export results tables
Writes filled versions of the CSV templates (new filenames; originals untouched).

In [ ]:
exports = {"descriptive_statistics_filled.csv": desc, "reliability_filled.csv": rel}
if HAVE_SM and "reg_table" in dir() and "model" in dir():
    exports["regression_results_filled.csv"] = reg_table.round(4)
if "slopes_df" in dir() and isinstance(slopes_df, pd.DataFrame):
    exports["simple_slopes_filled.csv"] = slopes_df
exports["correlation_matrix_filled.csv"] = corr

for fname, df in exports.items():
    df.to_csv(os.path.join(DATA, fname), index=False)
    print("saved:", os.path.join(DATA, fname), df.shape)

## 12. Reproducibility notes
- Set a random seed if any simulation/split is added: `np.random.seed(42)`.
- Record package versions for the methods section.

In [ ]:
import sys
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__)
import scipy; print("scipy:", scipy.__version__)
if HAVE_SM: import statsmodels; print("statsmodels:", statsmodels.__version__)
print("\nEnd of pipeline. Replace raw_data_template.csv with real data and re-run all cells.")